# 📱 Phone IMU → LSL Bridge (Flutter App)

This notebook connects to a **Flutter sensor app** running on your phone, reads
accelerometer + gyroscope data via its HTTP `/stream` endpoint (NDJSON), and
republishes the data as an **LSL (Lab Streaming Layer)** stream.

**Pipeline:**
```
Phone sensors → Flutter app (sensors_plus) → HTTP /stream → This notebook → LSL Outlet → MNE / LabRecorder
```

**6 channels:** `accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z`

In [ ]:
import sys

!{sys.executable} -m pip install pylsl requests

## ⚙️ Configuration

Set `PHONE_IP` to `"auto"` to scan the local network for the Flutter app,
or enter your phone's LAN IP address manually.

Make sure your phone and computer are on the **same Wi-Fi network** and
the Flutter app has **Start HTTP** active.

In [ ]:
import json
import socket
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import requests
from pylsl import (
    StreamInfo,
    StreamOutlet,
    StreamInlet,
    local_clock,
    resolve_byprop,
)

# ──────────────────────────────────────────────
#  Set PHONE_IP to your phone's LAN IP, or
#  "auto" to scan the local subnet automatically.
#  The Flutter app must have "Start HTTP" active.
# ──────────────────────────────────────────────
PHONE_IP       = "auto"           # ← "auto" or e.g. "192.168.1.42"
PHONE_PORT     = 8080

# LSL stream metadata
STREAM_NAME    = "PhoneSensors"
STREAM_TYPE    = "Sensors"
SOURCE_ID      = "phone_sensor_stream"
NUM_CHANNELS   = 6
SAMPLE_RATE_HZ = 100          # nominal rate from Flutter app (~10 ms period)

# How long to record
RUN_SECONDS    = 10

CHANNEL_LABELS = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]

print(f"LSL stream : {STREAM_NAME} ({NUM_CHANNELS}ch @ {SAMPLE_RATE_HZ} Hz)")
print(f"Record for : {RUN_SECONDS}s")

## 🔎 Discover the Flutter App

If `PHONE_IP` is `"auto"`, scan the local subnet for an HTTP server
responding on the configured port with a valid `/health` JSON payload.
Otherwise, use the manually specified IP.

In [ ]:
def _get_local_subnet_prefix() -> str:
    """Return the first 3 octets of the machine's LAN IP (e.g. '192.168.1')."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))  # doesn't actually send traffic
        ip = s.getsockname()[0]
        s.close()
        return ".".join(ip.split(".")[:3])
    except Exception:
        return "192.168.1"


def _probe(ip: str, port: int, timeout: float = 0.4) -> str | None:
    """Try to hit /health on ip:port; return the IP if it responds correctly."""
    try:
        r = requests.get(f"http://{ip}:{port}/health", timeout=timeout)
        if r.ok and "status" in r.json():
            return ip
    except Exception:
        pass
    return None


def discover_phone(port: int) -> str:
    """Scan the local /24 subnet for the Flutter HTTP server."""
    prefix = _get_local_subnet_prefix()
    print(f"Scanning {prefix}.1-254 on port {port} ...")

    with ThreadPoolExecutor(max_workers=64) as pool:
        futures = {
            pool.submit(_probe, f"{prefix}.{i}", port): i
            for i in range(1, 255)
        }
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                # Cancel remaining futures
                for f in futures:
                    f.cancel()
                return result

    raise RuntimeError(
        f"No Flutter app found on {prefix}.0/24 port {port}.\n"
        "Make sure the app is running with HTTP started and you're on the same network."
    )


# Resolve the phone IP
if PHONE_IP.lower() == "auto":
    PHONE_IP = discover_phone(PHONE_PORT)
    print(f"✅ Found Flutter app at {PHONE_IP}")
else:
    print(f"Using manually configured IP: {PHONE_IP}")

STREAM_URL = f"http://{PHONE_IP}:{PHONE_PORT}/stream"
SAMPLE_URL = f"http://{PHONE_IP}:{PHONE_PORT}/sample"
HEALTH_URL = f"http://{PHONE_IP}:{PHONE_PORT}/health"

print(f"Stream URL : {STREAM_URL}")
print(f"Sample URL : {SAMPLE_URL}")
print(f"Health URL : {HEALTH_URL}")

## 🔍 Health Check

Verify the phone is reachable and the HTTP server is responding.

In [ ]:
try:
    resp = requests.get(HEALTH_URL, timeout=3)
    resp.raise_for_status()
    info = resp.json()
    print("✅ Phone is reachable!")
    for k, v in info.items():
        print(f"   {k}: {v}")
except requests.RequestException as exc:
    print(f"❌ Cannot reach phone: {exc}")
    print("   Make sure the Flutter app has 'Start HTTP' active and you're on the same network.")

## 🧪 Test: Fetch a Single Sample

Quick sanity check — grab one sample from `/sample` to confirm the data format.

In [ ]:
try:
    resp = requests.get(SAMPLE_URL, timeout=3)
    resp.raise_for_status()
    data = resp.json()
    print("Raw JSON from /sample:")
    print(json.dumps(data, indent=2))
    print()

    a = data["accel"]
    g = data["gyro"]
    print(f"  accel  → x={a['x']:+.4f}  y={a['y']:+.4f}  z={a['z']:+.4f}")
    print(f"  gyro   → x={g['x']:+.4f}  y={g['y']:+.4f}  z={g['z']:+.4f}")
    print(f"  seq    → {data.get('sequence', '?')}")
    print("\n✅ Data format looks good!")
except Exception as exc:
    print(f"❌ Failed to fetch sample: {exc}")

## 📡 Sender + Receiver

- **Sender thread**: connects to the Flutter app's `/stream` endpoint (NDJSON),
  parses each JSON line, and pushes the 6-channel sample into an LSL outlet.
  Automatically reconnects on connection drops.
- **Receiver**: discovers the LSL stream and pulls samples for `RUN_SECONDS`.

In [ ]:
def parse_sample(data: dict) -> list:
    """Extract the 6 sensor channels from a Flutter app JSON packet.

    Expected format:
        {"timestamp": ..., "sequence": ...,
         "accel": {"x": ..., "y": ..., "z": ...},
         "gyro":  {"x": ..., "y": ..., "z": ...}}

    Returns [accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z]
    """
    a = data["accel"]
    g = data["gyro"]
    return [
        float(a["x"]), float(a["y"]), float(a["z"]),
        float(g["x"]), float(g["y"]), float(g["z"]),
    ]


def make_lsl_outlet() -> StreamOutlet:
    """Create an LSL StreamOutlet with proper channel metadata."""
    info = StreamInfo(
        STREAM_NAME, STREAM_TYPE,
        NUM_CHANNELS, SAMPLE_RATE_HZ,
        "float32", SOURCE_ID,
    )

    # Add channel metadata
    channels = info.desc().append_child("channels")
    for label in CHANNEL_LABELS:
        ch = channels.append_child("channel")
        ch.append_child_value("label", label)
        ch.append_child_value("type", "misc")
        ch.append_child_value("unit", "none")

    return StreamOutlet(info)


def sender(stop_event: threading.Event, max_retries: int = 5):
    """Connect to the Flutter app /stream (NDJSON) and push samples to LSL.

    Automatically retries on connection drops with exponential backoff.
    """
    outlet = make_lsl_outlet()
    pushed = 0
    retries = 0

    while not stop_event.is_set() and retries < max_retries:
        print(f"Sender: connecting to {STREAM_URL} ...")
        try:
            with requests.get(STREAM_URL, stream=True, timeout=10) as resp:
                resp.raise_for_status()
                retries = 0  # reset on successful connect
                print("Sender: connected — pushing to LSL.")

                for line in resp.iter_lines(decode_unicode=True):
                    if stop_event.is_set():
                        break
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                        sample = parse_sample(data)
                        outlet.push_sample(sample)
                        pushed += 1
                        if pushed % 500 == 0:
                            print(f"Sender: {pushed} samples pushed to LSL")
                    except (json.JSONDecodeError, KeyError, ValueError) as exc:
                        print(f"Sender: bad packet — {exc}")

        except requests.RequestException as exc:
            retries += 1
            backoff = min(2 ** retries, 10)
            print(f"Sender: connection error — {exc}")
            if retries < max_retries and not stop_event.is_set():
                print(f"Sender: retrying in {backoff}s (attempt {retries}/{max_retries})")
                stop_event.wait(backoff)

    print(f"Sender: stopped after pushing {pushed} samples.")


def receiver(duration_seconds: int = RUN_SECONDS):
    """Discover the LSL stream and pull samples for the given duration."""
    print(f"Receiver: resolving '{STREAM_TYPE}' stream ...")
    streams = resolve_byprop("type", STREAM_TYPE, timeout=10)

    if not streams:
        print("❌ No LSL stream found. Is the sender running?")
        return []

    inlet = StreamInlet(streams[0])
    print(f"Receiver: connected to '{streams[0].name()}' ({streams[0].channel_count()}ch).")

    samples = []
    start_real = datetime.now()
    start_lsl  = local_clock()
    stop_at    = time.monotonic() + duration_seconds

    while time.monotonic() < stop_at:
        sample, ts = inlet.pull_sample(timeout=1)
        if sample is None:
            continue

        wall_time = start_real + timedelta(seconds=ts - start_lsl)
        readable  = wall_time.strftime("%H:%M:%S.%f")[:-3]
        samples.append((wall_time, sample))
        if len(samples) <= 5 or len(samples) % 100 == 0:
            print(
                f"{readable}  "
                f"acc=({sample[0]:+7.3f}, {sample[1]:+7.3f}, {sample[2]:+7.3f})  "
                f"gyr=({sample[3]:+7.4f}, {sample[4]:+7.4f}, {sample[5]:+7.4f})"
            )

    print(f"\nReceiver: collected {len(samples)} samples in {duration_seconds}s.")
    return samples

## ▶️ Run the Pipeline

Start the sender thread, wait a moment for the LSL outlet to register,
then run the receiver for `RUN_SECONDS`.

In [ ]:
stop_event = threading.Event()
sender_thread = threading.Thread(target=sender, args=(stop_event,), daemon=True)

try:
    sender_thread.start()
    time.sleep(2)          # give the outlet time to register on the LSL network
    samples = receiver()
finally:
    stop_event.set()
    sender_thread.join(timeout=5)
    print("Pipeline stopped.")

## 📊 Quick Results Summary

In [ ]:
if samples:
    import statistics

    duration = (samples[-1][0] - samples[0][0]).total_seconds()
    actual_rate = (len(samples) - 1) / duration if duration > 0 else 0

    print(f"Total samples : {len(samples)}")
    print(f"Duration      : {duration:.2f} s")
    print(f"Actual rate   : {actual_rate:.1f} Hz  (nominal {SAMPLE_RATE_HZ} Hz)")
    print()

    for i, label in enumerate(CHANNEL_LABELS):
        vals = [s[1][i] for s in samples]
        print(
            f"  {label:>8s}  "
            f"mean={statistics.mean(vals):+8.4f}  "
            f"std={statistics.stdev(vals):7.4f}  "
            f"min={min(vals):+8.4f}  "
            f"max={max(vals):+8.4f}"
        )
else:
    print("No samples were collected.")

In [ ]:
import sys

!{sys.executable} -m pip install matplotlib numpy

## 📺 Live Continuous Stream Visualization

Connect directly to the Flutter app's `/stream` endpoint and display a
**continuously updating** pair of rolling charts (accelerometer + gyroscope).

- The plot refreshes ~5 times/sec with a rolling window.
- **Interrupt the kernel** (⬜ Stop button) to stop.
- Adjust `WINDOW_SECONDS` to control visible history.

In [ ]:
%matplotlib inline

import json
import threading
import time
from collections import deque

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import requests
from IPython.display import display, clear_output

# ── Settings ─────────────────────────────────────
WINDOW_SECONDS = 5
REFRESH_HZ     = 5
MAX_SAMPLES    = SAMPLE_RATE_HZ * WINDOW_SECONDS

# ── Thread-safe ring buffers ─────────────────────
_bufs = {k: deque(maxlen=MAX_SAMPLES) for k in ['t','ax','ay','az','gx','gy','gz']}
_n = 0
_lock = threading.Lock()

def _stream_reader(stop_evt):
    global _n
    t0 = time.monotonic()
    while not stop_evt.is_set():
        try:
            with requests.get(STREAM_URL, stream=True, timeout=10) as resp:
                resp.raise_for_status()
                for line in resp.iter_lines(decode_unicode=True):
                    if stop_evt.is_set(): return
                    if not line: continue
                    try:
                        d = json.loads(line)
                        a, g = d['accel'], d['gyro']
                        with _lock:
                            _bufs['t'].append(time.monotonic() - t0)
                            _bufs['ax'].append(float(a['x']))
                            _bufs['ay'].append(float(a['y']))
                            _bufs['az'].append(float(a['z']))
                            _bufs['gx'].append(float(g['x']))
                            _bufs['gy'].append(float(g['y']))
                            _bufs['gz'].append(float(g['z']))
                            _n += 1
                    except (json.JSONDecodeError, KeyError, ValueError):
                        pass
        except requests.RequestException:
            if not stop_evt.is_set(): stop_evt.wait(1)

# ── Start reader ────────────────────────────────
_stop = threading.Event()
_reader = threading.Thread(target=_stream_reader, args=(_stop,), daemon=True)
_reader.start()
print('⏳ Waiting for first samples ...')
while _n == 0 and not _stop.is_set(): time.sleep(0.1)
print(f'✅ Receiving — drawing live plot (interrupt kernel to stop)\n')

# ── Colors ───────────────────────────────────────
AC = ['#ef4444','#22c55e','#3b82f6']
GC = ['#f97316','#a855f7','#06b6d4']
BG, GR, TX = '#0f172a', '#1e293b', '#e2e8f0'

try:
    while True:
        with _lock:
            t = np.array(_bufs['t']); n = _n
            vals = {k: np.array(v) for k, v in _bufs.items() if k != 't'}
        if len(t) < 2:
            time.sleep(1/REFRESH_HZ); continue
        tr = t - t[-1]  # relative: 0=now, negative=past

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), dpi=100,
                                     facecolor=BG, sharex=True)
        fig.subplots_adjust(hspace=0.08, left=0.08, right=0.97, top=0.92, bottom=0.10)
        for sp in (a1, a2):
            sp.set_facecolor(BG)
            sp.tick_params(colors=TX, labelsize=9)
            for s in sp.spines.values(): s.set_color(GR)
            sp.grid(True, color=GR, lw=0.5, alpha=0.6)

        for arr, c, lb in zip([vals['ax'],vals['ay'],vals['az']], AC, 'XYZ'):
            a1.plot(tr, arr, color=c, lw=1, alpha=0.9, label=lb)
        a1.set_ylabel('Accel (m/s²)', color=TX, fontsize=10, fontweight='bold')
        a1.legend(loc='upper left', fontsize=8, framealpha=0.3,
                  labelcolor=TX, facecolor=BG, edgecolor=GR)

        for arr, c, lb in zip([vals['gx'],vals['gy'],vals['gz']], GC, 'XYZ'):
            a2.plot(tr, arr, color=c, lw=1, alpha=0.9, label=lb)
        a2.set_ylabel('Gyro (rad/s)', color=TX, fontsize=10, fontweight='bold')
        a2.set_xlabel('Time (s)', color=TX, fontsize=10)
        a2.legend(loc='upper left', fontsize=8, framealpha=0.3,
                  labelcolor=TX, facecolor=BG, edgecolor=GR)
        a2.set_xlim(-WINDOW_SECONDS, 0)
        a2.xaxis.set_major_locator(ticker.MultipleLocator(1))

        rate = n / t[-1] if t[-1] > 0 else 0
        fig.suptitle(
            f'📡 Live Sensor Stream  |  {n:,} samples  |  '
            f'{rate:.0f} Hz  |  {t[-1]:.1f}s elapsed',
            color=TX, fontsize=11, fontweight='bold')
        plt.show()
        time.sleep(1/REFRESH_HZ)

except KeyboardInterrupt:
    _stop.set()
    _reader.join(timeout=3)
    print('\n🛑 Live view stopped.')
    print(f'   Total samples: {_n:,}')
    if _bufs['t']:
        print(f'   Total time: {_bufs["t"][-1]:.1f}s')